In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox
import time
import random

# ---------------------------------------------------
# STRING MATCHING ALGORITHMS
# ---------------------------------------------------

def naive_search(text, pattern):
    n = len(text)
    m = len(pattern)

    matches = []
    comparisons = 0

    for i in range(n - m + 1):
        j = 0

        while j < m:
            comparisons += 1

            if text[i + j] != pattern[j]:
                break

            j += 1

        if j == m:
            matches.append(i)

    return matches, comparisons


def compute_lps(pattern):

    m = len(pattern)
    lps = [0] * m

    length = 0
    i = 1

    while i < m:

        if pattern[i] == pattern[length]:
            length += 1
            lps[i] = length
            i += 1

        else:

            if length != 0:
                length = lps[length - 1]

            else:
                lps[i] = 0
                i += 1

    return lps


def kmp_search(text, pattern):

    n = len(text)
    m = len(pattern)

    lps = compute_lps(pattern)

    i = 0
    j = 0

    matches = []
    comparisons = 0

    while i < n:

        comparisons += 1

        if pattern[j] == text[i]:
            i += 1
            j += 1

        if j == m:
            matches.append(i - j)
            j = lps[j - 1]

        elif i < n and pattern[j] != text[i]:

            if j != 0:
                j = lps[j - 1]

            else:
                i += 1

    return matches, comparisons


def rabin_karp(text, pattern, q=101):

    n = len(text)
    m = len(pattern)

    d = 256

    h = pow(d, m - 1, q)

    p_hash = 0
    t_hash = 0

    comparisons = 0
    matches = []

    for i in range(m):
        p_hash = (d * p_hash + ord(pattern[i])) % q
        t_hash = (d * t_hash + ord(text[i])) % q

    for s in range(n - m + 1):

        if p_hash == t_hash:

            for k in range(m):

                comparisons += 1

                if text[s + k] != pattern[k]:
                    break

            else:
                matches.append(s)

        if s < n - m:

            t_hash = (
                d * (t_hash - ord(text[s]) * h)
                + ord(text[s + m])
            ) % q

            if t_hash < 0:
                t_hash += q

    return matches, comparisons


# ---------------------------------------------------
# GUI FUNCTIONS
# ---------------------------------------------------

def random_text():

    letters = "ABCD"

    txt = "".join(random.choice(letters) for _ in range(100))

    text_box.delete("1.0", tk.END)
    text_box.insert(tk.END, txt)

    pattern_entry.delete(0, tk.END)
    pattern_entry.insert(0, "ABCD")


def clear_all():

    text_box.delete("1.0", tk.END)

    pattern_entry.delete(0, tk.END)

    naive_result.config(text="")

    kmp_result.config(text="")

    rk_result.config(text="")

    status.config(text="Status : Cleared")


def run_algorithms():

    text = text_box.get("1.0", tk.END).strip()

    pattern = pattern_entry.get()

    if text == "" or pattern == "":
        messagebox.showerror(
            "Error",
            "Please enter Text and Pattern"
        )
        return

    # ---------------- Naive ----------------

    start = time.perf_counter()

    matches, comp = naive_search(text, pattern)

    end = time.perf_counter()

    naive_time = (end - start) * 1000

    naive_result.config(
        text=f"""
Matches      : {matches}

Comparisons  : {comp}

Time         : {naive_time:.5f} ms
"""
    )

    # ---------------- KMP ----------------

    start = time.perf_counter()

    matches2, comp2 = kmp_search(text, pattern)

    end = time.perf_counter()

    kmp_time = (end - start) * 1000

    kmp_result.config(
        text=f"""
Matches      : {matches2}

Comparisons  : {comp2}

Time         : {kmp_time:.5f} ms
"""
    )
    # ---------------- Rabin-Karp ----------------

    start = time.perf_counter()

    matches3, comp3 = rabin_karp(text, pattern)

    end = time.perf_counter()

    rk_time = (end - start) * 1000

    rk_result.config(
        text=f"""
Matches      : {matches3}

Comparisons  : {comp3}

Time         : {rk_time:.5f} ms
"""
    )

    # -------- Performance Table --------

    for item in table.get_children():
        table.delete(item)

    table.insert(
        "",
        "end",
        values=("Naive", comp, f"{naive_time:.5f}")
    )

    table.insert(
        "",
        "end",
        values=("KMP", comp2, f"{kmp_time:.5f}")
    )

    table.insert(
        "",
        "end",
        values=("Rabin-Karp", comp3, f"{rk_time:.5f}")
    )

    status.config(text="Status : Algorithms Executed Successfully")


# ---------------------------------------------------
# MAIN WINDOW
# ---------------------------------------------------

root = tk.Tk()

root.title("String Matching Algorithm Visualizer")

root.geometry("1000x760")

root.configure(bg="#eef4ff")

title = tk.Label(
    root,
    text="STRING MATCHING ALGORITHM VISUALIZER",
    font=("Segoe UI", 20, "bold"),
    bg="#4a6fa5",
    fg="white",
    pady=12
)

title.pack(fill="x", pady=(0, 10))


frame = tk.Frame(root, bg="#eef4ff")
frame.pack(fill="both", expand=True, padx=20)


tk.Label(
    frame,
    text="Enter Text",
    font=("Segoe UI", 12, "bold"),
    bg="#eef4ff"
).pack(anchor="w")

text_box = tk.Text(
    frame,
    height=6,
    font=("Consolas", 11),
    relief="solid",
    bd=1
)

text_box.pack(fill="x", pady=5)


tk.Label(
    frame,
    text="Pattern",
    font=("Segoe UI", 12, "bold"),
    bg="#eef4ff"
).pack(anchor="w", pady=(10, 0))

pattern_entry = tk.Entry(
    frame,
    font=("Consolas", 12)
)

pattern_entry.pack(fill="x", pady=5)


button_frame = tk.Frame(frame, bg="#eef4ff")
button_frame.pack(pady=10)


tk.Button(
    button_frame,
    text="Generate Random",
    command=random_text,
    bg="#2196F3",
    fg="white",
    width=18
).grid(row=0, column=0, padx=5)


tk.Button(
    button_frame,
    text="Run Algorithms",
    command=run_algorithms,
    bg="#4CAF50",
    fg="white",
    width=18
).grid(row=0, column=1, padx=5)


tk.Button(
    button_frame,
    text="Clear",
    command=clear_all,
    bg="#f44336",
    fg="white",
    width=18
).grid(row=0, column=2, padx=5)


result_frame = tk.Frame(frame, bg="#eef4ff")
result_frame.pack(fill="x", pady=10)


naive_box = tk.LabelFrame(
    result_frame,
    text="Naive Algorithm",
    font=("Segoe UI", 11, "bold"),
    padx=10,
    pady=10
)

naive_box.grid(row=0, column=0, padx=8)

naive_result = tk.Label(
    naive_box,
    justify="left",
    font=("Consolas", 10)
)

naive_result.pack()


kmp_box = tk.LabelFrame(
    result_frame,
    text="KMP Algorithm",
    font=("Segoe UI", 11, "bold"),
    padx=10,
    pady=10
)

kmp_box.grid(row=0, column=1, padx=8)

kmp_result = tk.Label(
    kmp_box,
    justify="left",
    font=("Consolas", 10)
)

kmp_result.pack()


rk_box = tk.LabelFrame(
    result_frame,
    text="Rabin-Karp Algorithm",
    font=("Segoe UI", 11, "bold"),
    padx=10,
    pady=10
)

rk_box.grid(row=0, column=2, padx=8)

rk_result = tk.Label(
    rk_box,
    justify="left",
    font=("Consolas", 10)
)

rk_result.pack()


tk.Label(
    frame,
    text="Performance Comparison",
    font=("Segoe UI", 13, "bold"),
    bg="#eef4ff"
).pack(pady=8)


table = ttk.Treeview(
    frame,
    columns=("Algorithm", "Comparisons", "Time"),
    show="headings",
    height=5
)

table.heading("Algorithm", text="Algorithm")
table.heading("Comparisons", text="Comparisons")
table.heading("Time", text="Execution Time (ms)")

table.column("Algorithm", width=180)
table.column("Comparisons", width=150)
table.column("Time", width=180)

table.pack(fill="x")


status = tk.Label(
    root,
    text="Status : Ready",
    bg="#4a6fa5",
    fg="white",
    anchor="w",
    padx=10,
    pady=8
)

status.pack(fill="x")
# ---------------------------------------------------
# DEFAULT SAMPLE INPUT
# ---------------------------------------------------

sample_text = "AABAACAADAABAABA"
sample_pattern = "AABA"

text_box.insert("1.0", sample_text)
pattern_entry.insert(0, sample_pattern)

status.config(
    text="Status : Ready (Sample text loaded)"
)

# ---------------------------------------------------
# ABOUT MENU
# ---------------------------------------------------

def show_about():
    messagebox.showinfo(
        "About",
        "STRING MATCHING ALGORITHM VISUALIZER\n\n"
        "Design and Analysis of Algorithms Mini Project\n\n"
        "Algorithms Implemented:\n"
        "• Naive String Matching\n"
        "• Knuth-Morris-Pratt (KMP)\n"
        "• Rabin-Karp\n\n"
        "Features:\n"
        "✓ Match Positions\n"
        "✓ Number of Comparisons\n"
        "✓ Execution Time Comparison\n"
        "✓ Random Text Generator\n"
    )

menubar = tk.Menu(root)

help_menu = tk.Menu(
    menubar,
    tearoff=0
)

help_menu.add_command(
    label="About",
    command=show_about
)

menubar.add_cascade(
    label="Help",
    menu=help_menu
)

root.config(menu=menubar)

# ---------------------------------------------------
# START APPLICATION
# ---------------------------------------------------

root.mainloop()